# SPx - Spectral Processing & Unmixing

This notebook provides an interactive workflow for the SPx two-stage pipeline:

1. **Processing** - Read CSV spectral files, filter by wavelength thresholds, extract convex hull quotient features, generate statistics and plots.
2. **Unmixing** - Apply linear spectral unmixing (LSUMx) using reference endmembers to estimate mixture coefficients.

In [1]:
%pip install -e ".[notebook,env]"
# Restart the kernel after running this cell for the first time.

Obtaining file:///Users/liubomyrgavryliv/Repos/mineralogy-rocks/SPx
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 9.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 9.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 7.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 9.5 MB/s eta 0:00:00 0:00:01m
  Building editable for SPx (pyproject.toml) ... done
  Created wheel for SPx: filename=spx-1-0.editable-py3-none-any.whl size=1599 sha256=739e6b5050d52ff1333758b981da0b9a7f791c084c0ef59e0960b73859b5818b
  Stored in directory: /private/var/folders/3h/77l7ld2j7bbbx9jgfhfn580r0000gn/T/pip-ephem-wh

In [2]:
%matplotlib inline

import logging

from src.config import settings
from src.base.main import run_pipeline
from src.base.predict import run_prediction
from src.choices import THRESHOLDS

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

## Configuration

Set the paths below to point to your project directory. The input folder should contain CSV files with spectral data (wavelength + reflectance columns).

In [3]:
PROJECT_DIRECTORY = "/path/to/your/project"
INPUT_FOLDER = "input"
OUTPUT_FOLDER = "output"

settings.configure(PROJECT_DIRECTORY, INPUT_FOLDER, OUTPUT_FOLDER)

print(f"Project:  {settings.PROJECT_DIRECTORY}")
print(f"Input:    {settings.INPUT_PATH}")
print(f"Output:   {settings.OUTPUT_PATH}")

Project:  /Users/liubomyrgavryliv/Repos/mineralogy-rocks/SPx-test
Input:    /Users/liubomyrgavryliv/Repos/mineralogy-rocks/SPx-test/input
Output:   /Users/liubomyrgavryliv/Repos/mineralogy-rocks/SPx-test/output


## Thresholds

Spectral peak wavelength ranges (nm) used to filter input data. Edit the list below to customize.

| Peak | Range (nm) |
|------|------------|
| peak-1 | 5500 - 8000 |
| peak-2 | 4600 - 5540 |
| peak-3 | 4310 - 4788 |

In [5]:
thresholds = [
    ("peak-1", (5500, 8000)),
    ("peak-2", (4600, 5540)),
    ("peak-3", (4310, 4788)),
]

## Processing

Runs the full processing pipeline: generates spectra from CSV input, extracts features, computes statistics, and saves plots to the output directory. Plots are saved to disk in `output/plots/`.

In [6]:
run_pipeline(show_plots=False, thresholds=thresholds)

2026-03-06 10:30:35,658 INFO Starting NIR spectra processing
Generating spectra: 100%|██████████| 6/6 [00:00<00:00, 122.71file/s]
2026-03-06 10:30:35,754 INFO Found 18 spectra files.
Exporting continuum removed spectra: 100%|██████████| 18/18 [00:01<00:00, 10.10spectrum/s]
2026-03-06 10:30:43,969 INFO Processing completed successfully


<Figure size 4200x3300 with 0 Axes>

## Unmixing

Linear spectral unmixing using two reference endmembers. Set `ENDMEMBERS_PATH` to a custom `.xlsx` file, or leave as `None` to use the default `data/endmembers.xlsx`.

The endmembers file must contain exactly 2 rows (one per endmember) with matching parameter columns to the results.

In [ ]:
ENDMEMBERS_PATH = None

In [ ]:
run_prediction(endmembers_path=ENDMEMBERS_PATH)

## Results

Load and display the processing and prediction results.

In [7]:
import pandas as pd

results_path = settings.OUTPUT_PATH / "data" / "results.xlsx"
predicted_path = settings.OUTPUT_PATH / "data" / "results_predicted.xlsx"

if results_path.exists():
    print("Processing results:")
    display(pd.read_excel(results_path))

if predicted_path.exists():
    print("\nPrediction results:")
    display(pd.read_excel(predicted_path))

Processing results:


,peak,filename,cstart_wvl,cstop_wvl,abs_wvl,abs_depth,area,cslope,FWHM_y,FWHM_delta,...,FW,FW_left_width,FW_right_width,FW_assymetry,FWHM_left_width,FWHM_right_width,FWHM_assymetry,D,E,E*
0,1,6B_NIR DRIFT_1-peak-1,5505.860,7995.549,7066.014,0.954404,19.870618,-2.581808e-06,0.977202,219.848,...,2489.689,1560.154,929.535,1.678424,106.067,113.781,0.932203,0.045596,54603.317260,4821.658485
1,2,6B_NIR DRIFT_1-peak-2,4699.748,5411.363,5239.727,0.898568,19.705648,1.788889e-07,0.949284,148.494,...,711.615,539.979,171.636,3.146071,100.282,48.212,2.080022,0.101432,7015.680031,1463.974748
2,3,6B_NIR DRIFT_1-peak-3,4408.545,4697.819,4530.040,0.953861,4.251197,4.937533e-06,0.976931,71.354,...,289.274,121.495,167.779,0.724137,21.213,50.141,0.423067,0.046139,6269.661228,1546.510946
3,1,6B_NIR DRIFT_2-peak-1,5505.860,7995.549,7071.799,0.956949,17.787350,-2.615467e-06,0.978475,208.277,...,2489.689,1565.939,923.750,1.695198,100.281,107.996,0.928562,0.043051,57831.325661,4837.927554
4,2,6B_NIR DRIFT_2-peak-2,4707.462,5411.363,5239.727,0.906801,17.285435,2.514558e-08,0.953400,140.781,...,703.901,532.265,171.636,3.101127,94.497,46.284,2.041677,0.093199,7552.659718,1510.540527
5,3,6B_NIR DRIFT_2-peak-3,4406.616,4692.034,4530.040,0.953764,4.358452,5.436588e-06,0.976882,75.211,...,285.418,123.424,161.994,0.761905,21.213,53.998,0.392848,0.046236,6173.130874,1626.692592
6,1,6B_NIR DRIFT_3-peak-1,5505.860,7983.979,7073.728,0.945648,27.335946,-2.721782e-06,0.972824,233.348,...,2478.119,1567.868,910.251,1.722457,127.281,106.067,1.200006,0.054352,45594.142167,4293.297411
7,2,6B_NIR DRIFT_3-peak-2,4701.676,5411.363,5239.727,0.887294,22.042598,-1.355527e-08,0.943647,152.351,...,709.687,538.051,171.636,3.134838,104.139,48.212,2.160022,0.112706,6296.799215,1351.755996
8,3,6B_NIR DRIFT_3-peak-3,4406.616,4690.105,4530.040,0.950354,4.570831,3.967879e-06,0.975177,69.425,...,283.489,123.424,160.065,0.771087,21.213,48.212,0.439994,0.049646,5710.247387,1398.410255
9,1,ELu_Bulk_1-peak-1,5951.343,7798.843,7067.942,0.774150,93.676200,-1.846766e-06,0.887075,269.990,...,1847.500,1116.599,730.901,1.527702,156.208,113.782,1.372871,0.225850,8180.200248,1195.438303
